# Fruits results Analysis

In [ ]:
"""
Fruit Distribution Analysis — Success vs Failure
==================================================
Analyzes the repartition of target fruits across grasp outcomes
(normal_success / slip_success / fail) from a robotics manipulation
experiment metadata file.

Data schema expected:
    {
        "<episode_id>": {
            "result": "normal_success" | "slip_success" | "fail",
            "domain_rand_params": {
                "target_fruit": "<fruit_name>",
                ...
            },
            ...
        },
        ...
    }
"""

import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

# ── 1. Load data ──────────────────────────────────────────────────────────────

DATA_PATH = Path("/home/pierre/Documents/pick_and_place_fruits/demo/metadata.json")

with open(DATA_PATH) as f:
    data = json.load(f)

episodes = list(data.values())
N = len(episodes)

# ── 2. Build flat table ───────────────────────────────────────────────────────

results = np.array([e["result"] for e in episodes])
fruits  = np.array([e["domain_rand_params"]["target_fruit"] for e in episodes])

fruit_labels  = sorted(set(fruits))
result_labels = ["normal_success", "slip_success", "fail"]
COLORS = {
    "normal_success": "#4CAF50",
    "slip_success"  : "#2196F3",
    "fail"          : "#E53935",
}
DISPLAY = {
    "normal_success": "Normal success",
    "slip_success"  : "Slip success",
    "fail"          : "Fail",
}

# ── 3. Counts ─────────────────────────────────────────────────────────────────

counts = {
    res: Counter(fruits[results == res])
    for res in result_labels
}
abs_counts = {
    res: np.array([counts[res].get(f, 0) for f in fruit_labels])
    for res in result_labels
}

total_per_fruit  = np.array([Counter(fruits)[f] for f in fruit_labels])
combined_success = abs_counts["normal_success"] + abs_counts["slip_success"]
success_rate     = combined_success / total_per_fruit * 100

overall = Counter(results)
n_normal  = overall["normal_success"]
n_slip    = overall["slip_success"]
n_success = n_normal + n_slip
n_fail    = overall["fail"]

# ── 4. Print summary table ────────────────────────────────────────────────────

print("=" * 74)
print(f"{'FRUIT DISTRIBUTION — SUCCESS vs FAILURE':^74}")
print("=" * 74)
print(f"  Total episodes  : {N}")
print(f"  Normal success  : {n_normal}  ({n_normal/N*100:.1f} %)")
print(f"  Slip   success  : {n_slip}  ({n_slip/N*100:.1f} %)")
print(f"  Total  success  : {n_success}  ({n_success/N*100:.1f} %)")
print(f"  Fail            : {n_fail}  ({n_fail/N*100:.1f} %)")
print("-" * 74)
header = (f"{'Fruit':<12}  {'Total':>6}  {'Normal✓':>9}  "
          f"{'Slip✓':>7}  {'Fail':>6}  {'Success %':>10}")
print(header)
print("-" * 74)
for i, fruit in enumerate(fruit_labels):
    tot = total_per_fruit[i]
    ns  = abs_counts["normal_success"][i]
    ss  = abs_counts["slip_success"][i]
    fl  = abs_counts["fail"][i]
    pct = success_rate[i]
    print(f"  {fruit:<10}  {tot:>6}  {ns:>9}  {ss:>7}  {fl:>6}  {pct:>9.1f} %")
print("=" * 74)

# ── 5. Plots ──────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(18, 10))
fig.suptitle(
    "Target Fruit Distribution — Normal Success / Slip Success / Fail",
    fontsize=15, fontweight="bold", y=0.98,
)
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

x     = np.arange(len(fruit_labels))
bar_w = 0.25   # three bars per group

# ── 5a. Grouped bar — absolute counts (3 outcomes) ───────────────────────────
ax1 = fig.add_subplot(gs[0, :2])

offsets = [-bar_w, 0, bar_w]
for (res, off) in zip(result_labels, offsets):
    bars = ax1.bar(
        x + off, abs_counts[res], bar_w,
        label=DISPLAY[res], color=COLORS[res],
        edgecolor="white", linewidth=0.6,
    )
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax1.text(
                bar.get_x() + bar.get_width() / 2, h + 0.1,
                str(int(h)), ha="center", va="bottom", fontsize=8,
            )

ax1.set_xticks(x)
ax1.set_xticklabels(fruit_labels, fontsize=11)
ax1.set_ylabel("Episode count")
ax1.set_title("Absolute counts per fruit")
ax1.legend(framealpha=0.9)
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax1.grid(axis="y", linestyle="--", alpha=0.4)
ax1.set_axisbelow(True)

# ── 5b. Stacked 100 % bar (3 outcomes) ───────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])

normal_pct = abs_counts["normal_success"] / total_per_fruit * 100
slip_pct   = abs_counts["slip_success"]   / total_per_fruit * 100
fail_pct   = 100 - normal_pct - slip_pct

bottoms = np.zeros(len(fruit_labels))
for res, pcts in zip(result_labels, [normal_pct, slip_pct, fail_pct]):
    ax2.bar(fruit_labels, pcts, bottom=bottoms,
            color=COLORS[res], label=DISPLAY[res])
    for i, (p, b) in enumerate(zip(pcts, bottoms)):
        if p >= 8:
            ax2.text(i, b + p / 2, f"{p:.0f}%",
                     ha="center", va="center",
                     fontsize=8, color="white", fontweight="bold")
    bottoms += pcts

ax2.set_ylabel("Proportion (%)")
ax2.set_title("Outcome proportion\nper fruit (100 % stacked)")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax2.tick_params(axis="x", labelsize=9)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.set_axisbelow(True)

# ── 5c. Combined success rate bar ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])

bar_colors = [
    COLORS["normal_success"] if r >= 50 else COLORS["fail"]
    for r in success_rate
]
bars_sr = ax3.bar(fruit_labels, success_rate, color=bar_colors,
                  edgecolor="white", linewidth=0.6)
ax3.axhline(
    n_success / N * 100,
    color="steelblue", linestyle="--", linewidth=1.4,
    label=f"Overall avg ({n_success/N*100:.1f} %)",
)
for bar, val in zip(bars_sr, success_rate):
    ax3.text(bar.get_x() + bar.get_width() / 2, val + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

ax3.set_ylabel("Success rate (%)")
ax3.set_title("Per-fruit success rate (normal + slip)")
ax3.set_ylim(0, 110)
ax3.legend(framealpha=0.9)
ax3.grid(axis="y", linestyle="--", alpha=0.4)
ax3.set_axisbelow(True)

# ── 5d. Overall pie (3 wedges) ────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])

wedge_sizes  = [n_normal, n_slip, n_fail]
wedge_labels = [
    f"Normal ✓\n{n_normal} ({n_normal/N*100:.1f}%)",
    f"Slip ✓\n{n_slip} ({n_slip/N*100:.1f}%)",
    f"Fail\n{n_fail} ({n_fail/N*100:.1f}%)",
]
# Drop zero-count wedges to avoid matplotlib warnings
nonzero = [(s, l, COLORS[r])
           for s, l, r in zip(wedge_sizes, wedge_labels, result_labels) if s > 0]
sizes, labels, colors = zip(*nonzero)

ax4.pie(
    sizes, labels=labels, colors=colors,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    textprops={"fontsize": 9},
)
ax4.set_title("Overall outcome split")

out_path = Path("fruit_success_analysis.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nPlot saved → {out_path}")

                 FRUIT DISTRIBUTION — SUCCESS vs FAILURE                  
  Total episodes  : 88
  Normal success  : 18  (20.5 %)
  Slip   success  : 31  (35.2 %)
  Total  success  : 49  (55.7 %)
  Fail            : 39  (44.3 %)
--------------------------------------------------------------------------
Fruit          Total    Normal✓    Slip✓    Fail   Success %
--------------------------------------------------------------------------
  banana          17          6        2       9       47.1 %
  cherry          20          4        7       9       55.0 %
  lemon           16          1        5      10       37.5 %
  orange          15          3        4       8       46.7 %
  pear            20          4       13       3       85.0 %

Plot saved → out.png


# Put in Sorter Success Analysis 

In [5]:
"""
Shape Distribution Analysis — Success vs Failure
==================================================
Analyzes the repartition of target shapes across grasp outcomes
(normal_success / fail) from the TacEx shape-sorter evaluation results.

Data schema expected:
    {
        "<episode_id>": {
            "result": "normal_success" | "fail",
            "domain_rand_params": {
                "target_shape": "<shape_name>",
                "active_shapes": ["<shape>", ...]
            },
            "cost_step": <int>,
            "cost_time": <float>
        },
        ...
    }
"""

import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

# ── 1. Load data ──────────────────────────────────────────────────────────────

DATA_PATH = Path("/home/pierre/Documents/data_jz/put_smalls_shapes_sorter/demo/metadata.json")

with open(DATA_PATH) as f:
    data = json.load(f)

episodes = list(data.values())
N = len(episodes)

# ── 2. Build flat arrays ──────────────────────────────────────────────────────

results = np.array([e["result"] for e in episodes])
shapes  = np.array([e["domain_rand_params"]["target_shape"] for e in episodes])
steps   = np.array([e["cost_step"] for e in episodes], dtype=float)
n_distractors = np.array(
    [len(e["domain_rand_params"]["active_shapes"]) - 1 for e in episodes],
    dtype=int
)

# Normalise label: treat "normal_success" as "success"
results = np.where(results == "normal_success", "success", results)

shape_labels  = sorted(set(shapes))
result_labels = ["success", "fail"]
COLORS = {"success": "#4CAF50", "fail": "#E53935"}

# ── 3. Counts ─────────────────────────────────────────────────────────────────

counts = {res: Counter(shapes[results == res]) for res in result_labels}

abs_counts = {
    res: np.array([counts[res].get(s, 0) for s in shape_labels])
    for res in result_labels
}

total_per_shape = np.array([Counter(shapes)[s] for s in shape_labels])
success_rate    = abs_counts["success"] / total_per_shape * 100

overall = Counter(results)

# ── 4. Print summary table ────────────────────────────────────────────────────

print("=" * 62)
print(f"{'SHAPE DISTRIBUTION — SUCCESS vs FAILURE':^62}")
print("=" * 62)
print(f"  Total episodes : {N}")
print(f"  Success        : {overall['success']}  ({overall['success']/N*100:.1f} %)")
print(f"  Fail           : {overall['fail']}  ({overall['fail']/N*100:.1f} %)")
print("-" * 62)
header = f"{'Shape':<18}  {'Total':>6}  {'Success':>9}  {'Fail':>6}  {'Success %':>10}"
print(header)
print("-" * 62)
for i, shape in enumerate(shape_labels):
    tot = total_per_shape[i]
    s   = abs_counts["success"][i]
    f   = abs_counts["fail"][i]
    pct = success_rate[i]
    print(f"  {shape:<16}  {tot:>6}  {s:>9}  {f:>6}  {pct:>9.1f} %")
print("=" * 62)

# ── 5. Plots ──────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    "Target Shape Distribution — Success vs Failure (ACT Policy)",
    fontsize=15, fontweight="bold", y=0.98
)
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

x     = np.arange(len(shape_labels))
bar_w = 0.35

# ── 5a. Grouped bar — absolute counts ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])

bars_s = ax1.bar(x - bar_w/2, abs_counts["success"], bar_w,
                 label="Success", color=COLORS["success"], edgecolor="white", linewidth=0.6)
bars_f = ax1.bar(x + bar_w/2, abs_counts["fail"],    bar_w,
                 label="Fail",    color=COLORS["fail"],    edgecolor="white", linewidth=0.6)

for bar in list(bars_s) + list(bars_f):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

ax1.set_xticks(x)
ax1.set_xticklabels(shape_labels, fontsize=11)
ax1.set_ylabel("Episode count")
ax1.set_title("Absolute counts per shape")
ax1.legend(framealpha=0.9)
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax1.grid(axis="y", linestyle="--", alpha=0.4)
ax1.set_axisbelow(True)

# ── 5b. Stacked 100 % bar ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])

fail_rate = 100 - success_rate
ax2.bar(shape_labels, success_rate, color=COLORS["success"], label="Success")
ax2.bar(shape_labels, fail_rate,    bottom=success_rate,
        color=COLORS["fail"], label="Fail")

for i, (s, f) in enumerate(zip(success_rate, fail_rate)):
    if s >= 8:
        ax2.text(i, s / 2,     f"{s:.0f}%", ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold")
    if f >= 8:
        ax2.text(i, s + f / 2, f"{f:.0f}%", ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold")

ax2.set_ylabel("Proportion (%)")
ax2.set_title("Outcome proportion\nper shape (100 % stacked)")
ax2.set_ylim(0, 100)
ax2.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax2.tick_params(axis="x", labelsize=9, rotation=15)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.set_axisbelow(True)

# ── 5c. Per-shape success rate ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])

bar_colors = [
    COLORS["success"] if r >= 50 else COLORS["fail"]
    for r in success_rate
]
bars_sr = ax3.bar(shape_labels, success_rate, color=bar_colors,
                  edgecolor="white", linewidth=0.6)

overall_rate = overall["success"] / N * 100
ax3.axhline(overall_rate, color="steelblue", linestyle="--", linewidth=1.4,
            label=f"Overall avg ({overall_rate:.1f} %)")

for bar, val in zip(bars_sr, success_rate):
    ax3.text(bar.get_x() + bar.get_width() / 2, val + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

ax3.set_ylabel("Success rate (%)")
ax3.set_title("Per-shape success rate")
ax3.set_ylim(0, 110)
ax3.legend(framealpha=0.9)
ax3.grid(axis="y", linestyle="--", alpha=0.4)
ax3.set_axisbelow(True)

# ── 5d. Overall pie ───────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])

wedge_sizes  = [overall["success"], overall["fail"]]
wedge_labels = [
    f"Success\n{overall['success']} ({overall['success']/N*100:.1f}%)",
    f"Fail\n{overall['fail']} ({overall['fail']/N*100:.1f}%)",
]
ax4.pie(
    wedge_sizes,
    labels=wedge_labels,
    colors=[COLORS["success"], COLORS["fail"]],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    textprops={"fontsize": 9},
)
ax4.set_title("Overall outcome split")

out_path = Path("shape_success_analysis.png")
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nPlot saved → {out_path}")

           SHAPE DISTRIBUTION — SUCCESS vs FAILURE            
  Total episodes : 84
  Success        : 10  (11.9 %)
  Fail           : 74  (88.1 %)
--------------------------------------------------------------
Shape                Total    Success    Fail   Success %
--------------------------------------------------------------
  cube                  14          1      13        7.1 %
  cylinder              16          3      13       18.8 %
  moon                  12          0      12        0.0 %
  star                  23          2      21        8.7 %
  triangular_prism      19          4      15       21.1 %

Plot saved → shape_success_analysis.png


# Weights Sorting